In [ ]:
using ProgressBars
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

## 2 Harmonic oscillators

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
ω_sys_1=4.3*2*pi  # GHz
ω_sys_2=4*2*pi  # GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
Z_sys_1 = C_inv/ω_sys_1 # Characteristic impedance (dimensionless)
Z_sys_2 = C_inv/ω_sys_2 # Characteristic impedance (dimensionless)
κ=1/20 # 1/ns from Jeffrey et al.
g=25; # Coupling strength between the two oscillators

In [ ]:
n_huge=2000
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=90))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
n_exp=n

bath_params=(ω_c=50, ω_cut_off=100)
bath=Exp_decay
AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n_exp, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

### Mapping ###

ω = vcat(ω_sys_1,ω_sys_2,onsite_energies) # GHz
g_q=vcat(0,0, -hopping_energies) 
g_ϕ=vcat(-g*sqrt(Z_sys_1)*sqrt(Z_sys_2),-system_chain_coupling*sqrt(Z_sys_2), hopping_energies) 

ϕ_init = (0.0,0.0)
q_init = (1.0,1.0)

ϕ_0 = vcat(ϕ_init[1]/sqrt(Z_sys_1),ϕ_init[2]/sqrt(Z_sys_2),fill(0.0, n))
q_0 = vcat(q_init[1]*sqrt(Z_sys_1),q_init[2]*sqrt(Z_sys_2),fill(0.0, n))

Mq=SymTridiagonal(ω, g_q)
inv_Mq= inv(Mq)
Mϕ=SymTridiagonal(ω, g_ϕ)

A=Mq * Mϕ
vals, vecs = eigen(A)

ωk² = real(vals)
ωk = sqrt.(ωk²)
P = real(vecs)  # columns = normal modes

P_inv=inv(P)

# Project initial conditions into normal mode basis
ϕ_0_normal_modes = (P_inv * ϕ_0)
ϕ_dot_0_normal_modes = (P_inv * Mq * q_0)

tspan = (0.0, t_max)
dt=1/(20*maximum(ωk))
npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points

In [ ]:
# Time evolution in normal coordinates
tlist_bis = range(tspan[1], tspan[2], length=npoints)
ϕ_t_normal_modes_bis = [ϕ_0_normal_modes .* cos.(ωk .* t) .+ (ϕ_dot_0_normal_modes ./ ωk) .* sin.(ωk .* t) for t in tlist_bis]
ϕ_t_dot_normal_modes_bis= [ϕ_0_normal_modes .* (-ωk) .* sin.(ωk .* t) .+ (ϕ_dot_0_normal_modes) .* cos.(ωk .* t) for t in tlist_bis]
# Reconstruct real-space fields
ϕ_t_bis = [P * a for a in ϕ_t_normal_modes_bis]
ϕ_t_dot_bis = [P * a for a in ϕ_t_dot_normal_modes_bis] # \dot{ϕ} = P * \dot{ϕ}_normal_modes
q_t_bis = [inv_Mq * P * a for a in ϕ_t_dot_normal_modes_bis]  # q = \dot{ϕ}/Mq

ϕ_sys_1 =  [ϕ_t_bis[i][1] for i in 1:length(tlist_bis)]*sqrt(Z_sys_1)
q_sys_1 =  [q_t_bis[i][1] for i in 1:length(tlist_bis)]/ sqrt(Z_sys_1)

ϕ_sys_2 =  [ϕ_t_bis[i][2] for i in 1:length(tlist_bis)]*sqrt(Z_sys_2)
q_sys_2 =  [q_t_bis[i][2] for i in 1:length(tlist_bis)]/ sqrt(Z_sys_2)


In [ ]:
D1_bis= diag(P' * inv_Mq * P)
D2_bis= diag(P' * Mϕ * P);

In [ ]:
Energies_normal_modes_bis = [D1_bis[i]*getindex.(ϕ_t_dot_normal_modes_bis,i)[1]^2 + D2_bis[i]*getindex.(ϕ_t_normal_modes_bis,i)[1]^2 for i in 1:n+2]

In [ ]:
sum(Energies_normal_modes_bis)

In [ ]:
E_sys_bis[1]

In [ ]:
E_sys_bis= q_sys_1.^2*C_inv + ϕ_sys_1.^2*ω_sys_1^2/C_inv + q_sys_2.^2*C_inv + ϕ_sys_2.^2*ω_sys_2^2/C_inv + g*(ϕ_sys_1.*ϕ_sys_2)
E_sys_1= q_sys_1.^2*C_inv + ϕ_sys_1.^2*ω_sys_1^2/C_inv
E_sys_2= q_sys_2.^2*C_inv + ϕ_sys_2.^2*ω_sys_2^2/C_inv

In [ ]:
plot(ωk/2pi, (Energies_normal_modes_bis))

In [ ]:
E_sys_1[1]

In [ ]:
E_sys_1[end]

In [ ]:
P1_norm = sum(P[1,:].^2)
P2_norm = sum(P[2,:].^2)

In [ ]:
plot(ωk/2pi, (Energies_normal_modes_bis-E_sys_1[1]*P[1,:].^2/P1_norm-E_sys_2[1]*P[2,:].^2/P2_norm))
plot!(ωk/2pi, (Energies_normal_modes_bis-E_sys_1[end]*P[1,:].^2/P1_norm-E_sys_2[end]*P[2,:].^2/P2_norm))
plot!(xlims=(3.5,6.5))

In [ ]:
plot(ωk/2pi, Energies_normal_modes_bis-E_sys_1[end]*P[1,:].^2-E_sys_2[end]*P[2,:].^2)

In [ ]:
plot(ωk/2pi, P[1,:].^2/P1_norm, label="First oscillator")
plot!(ωk/2pi, P[2,:].^2/P2_norm, label="Second oscillator")
plot!(xlims=(3.5,4.5), ylabel="Coupling strength", xlabel="Frequency (GHz)")
#savefig("coupling_strengths_two_osc.pdf")

In [ ]:
25.8/4

In [ ]:
### Mapping ###
Z_sys = C_inv / ω_sys # Characteristic impedance in GHz
ω_chain = vcat(onsite_energies) # GHz
g_q_chain=vcat(-hopping_energies) 
g_ϕ_chain=vcat(hopping_energies) 

Mq_chain=SymTridiagonal(ω_chain, g_q_chain)
inv_Mq_chain= inv(Mq_chain)
Mϕ_chain=SymTridiagonal(ω_chain, g_ϕ_chain)

A_chain=Mq_chain * Mϕ_chain
vals_chain, vecs_chain = eigen(A_chain)

ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)
P_chain_inv=inv(P_chain)

In [ ]:
ϕ_t_chain_bis = @view ϕ_t_bis[1:npoints]  # shorten if needed
ϕ_t_dot_chain_bis = @view ϕ_t_dot_bis[1:npoints]
# Allocate result
Energies_normal_modes_chain_bis = Matrix{Float64}(undef, n, npoints)

# Compute S_chain and S_chain_dot on the fly, avoid storing large intermediates
for j in 1:npoints
    S = P_chain_inv * @view ϕ_t_chain_bis[j][3:n+2]
    S_dot = P_chain_inv * @view ϕ_t_dot_chain_bis[j][3:n+2]
    for i in 1:n
        Energies_normal_modes_chain_bis[i, j] = D1_chain[i] * S_dot[i]^2 + D2_chain[i] * S[i]^2
    end
end

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain_bis[:,end], label="")
plot!(ylabel="Energy stored in the chain\n normal modes", xlabel="Frequency (GHz)", xlims=(3.5,5), title = "t = $(t_max) ns, n=$(n)")
#savefig("Energies_normal_modes_chain_two_osc.pdf")

# 2 linear oscillators, reversing the chain mapping

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
ω_sys_1=4.3*2*pi  # GHz
ω_sys_2=4*2*pi  # GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
Z_sys_1 = C_inv/ω_sys_1 # Characteristic impedance (dimensionless)
Z_sys_2 = C_inv/ω_sys_2 # Characteristic impedance (dimensionless)
κ=1/20 # 1/ns from Jeffrey et al.
g=25; # Coupling strength between the two oscillators

In [ ]:
n_huge=2000
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=90))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 1/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n=n_opt+10;

In [ ]:
n_exp=n

bath_params=(ω_c=50, ω_cut_off=100)
bath=Exp_decay
AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n_exp, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

### Mapping ###

ω = vcat(ω_sys_1,ω_sys_2,onsite_energies) # GHz
g_q=vcat(0,0, -hopping_energies) 
g_ϕ=vcat(-g*sqrt(Z_sys_1)*sqrt(Z_sys_2),-system_chain_coupling*sqrt(Z_sys_2), hopping_energies) 

ϕ_init = (0.0,0.0)
q_init = (1.0,1.0)

ϕ_0 = vcat(ϕ_init[1]/sqrt(Z_sys_1),ϕ_init[2]/sqrt(Z_sys_2),fill(0.0, n))
q_0 = vcat(q_init[1]*sqrt(Z_sys_1),q_init[2]*sqrt(Z_sys_2),fill(0.0, n))

Mq=SymTridiagonal(ω, g_q)
inv_Mq= inv(Mq)
Mϕ=SymTridiagonal(ω, g_ϕ)

A=Mq * Mϕ
vals, vecs = eigen(A)

ωk² = real(vals)
ωk = sqrt.(ωk²)
P = real(vecs)  # columns = normal modes

P_inv=inv(P)

# Project initial conditions into normal mode basis
ϕ_0_normal_modes = (P_inv * ϕ_0)
ϕ_dot_0_normal_modes = (P_inv * Mq * q_0)

tspan = (0.0, t_max)
dt=1/(20*maximum(ωk))
npoints = Int(round((tspan[2]-tspan[1])/dt));  # number of time points

In [ ]:
# Time evolution in normal coordinates
tlist_bis = range(tspan[1], tspan[2], length=npoints)
ϕ_t_normal_modes_bis = [ϕ_0_normal_modes .* cos.(ωk .* t) .+ (ϕ_dot_0_normal_modes ./ ωk) .* sin.(ωk .* t) for t in tlist_bis]
ϕ_t_dot_normal_modes_bis= [ϕ_0_normal_modes .* (-ωk) .* sin.(ωk .* t) .+ (ϕ_dot_0_normal_modes) .* cos.(ωk .* t) for t in tlist_bis]
# Reconstruct real-space fields
ϕ_t_bis = [P * a for a in ϕ_t_normal_modes_bis]
ϕ_t_dot_bis = [P * a for a in ϕ_t_dot_normal_modes_bis]; # \dot{ϕ} = P * \dot{ϕ}_normal_modes

In [ ]:
Mq_chain = SymTridiagonal(onsite_energies, -hopping_energies)
Mϕ_chain = SymTridiagonal(onsite_energies, hopping_energies)
ϕ_chain_bis=[getindex.(ϕ_t_bis,i) for i in 3:n+2]
ϕ_dot_chain_bis=[getindex.(ϕ_t_dot_bis,i) for i in 3:n+2];

In [ ]:
ωk_chain_bis,Energies_normal_modes_chain_bis = emission_spectrum(ϕ_chain_bis, ϕ_dot_chain_bis, Mϕ_chain, Mq_chain)

In [ ]:
plot(ωk_chain_bis/2pi,Energies_normal_modes_chain_bis[end,:])
vline!([ω_sys_1/2pi],linestyle=:dash, color=:red)
vline!([ω_sys_2/2pi],linestyle=:dash, color=:red)
annotate!(ω_sys_1/2pi+.05,1e-10, Plots.text("mode a", 8, :red, rotation = 90 ))
annotate!(ω_sys_2/2pi+.05,1e-10, Plots.text("mode b", 8, :red, rotation = 90 ))
plot!(ylabel="Energy stored in the chain", 
xlabel="Frequency (GHz)", 
title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", 
legend=false,
xlims=(3,5),
yaxis=:log10)
savefig("Figures/Emission_spectrum/Energies_normal_modes_chain_two_osc.pdf")